#Cell 1

In [ ]:
# =============================================================================
# Thesis — GeoAI Café Site Selection · Milan
# Phase 2 · Notebook 3 · H3 Grid Generation and Spatial Backbone
# =============================================================================
#
# Purpose:
#   Generate the H3 hexagonal grid at resolution 9 covering the study area,
#   assign every Silver layer feature to its containing H3 cell, apply the
#   binary land use filter to exclude non-viable cells, and save the spatial
#   backbone GeoDataFrame that all subsequent feature engineering builds on.
#
# Key decisions documented here:
#   - H3 resolution 9: ~0.1 km² per cell, ~175m centroid-to-edge distance
#   - Viability filter: cells with less than 20% area overlap with viable Urban
#     Atlas zones are excluded from all downstream scoring and training
#   - Cell centroid in EPSG:32632 is the measurement anchor for all
#     proximity and density calculations in Notebook 4
#
# Inputs (all from Silver folder):
#   - silver_study_area.geojson
#   - silver_cafes.geojson
#   - silver_viable_mask.geojson
#
# Outputs (saved to Silver folder):
#   - silver_h3_grid.geojson        — full grid with viability flag
#   - silver_h3_viable.geojson      — viable cells only (analysis backbone)
#   - silver_cafe_cell_index.csv    — maps each café to its H3 cell ID
#   - silver_h3_log.txt
#
# CRS for analysis:    EPSG:32632 UTM Zone 32N
# CRS for H3 input:    EPSG:4326 WGS84 (H3 library requires lat/lon)
# CRS for export:      EPSG:4326 WGS84 (GeoJSON standard for web maps)
# =============================================================================

#Cell 2

In [ ]:
!pip install h3 -q
!pip install geopandas -q
!pip install shapely -q
!pip install folium -q
!pip install mapclassify -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 11.7 MB/s eta 0:00:00


#Cell 3

In [ ]:
import os
import datetime
import warnings

# Scoped warning filters — suppress only known noisy third-party warnings.
# UserWarning (including warnings.warn() call in assign_h3_cell for H3
# assignment failures) must NOT be suppressed.
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', module='osmnx')

import h3
import geopandas as gpd
import folium

from shapely.geometry import Polygon, mapping

from google.colab import drive
drive.mount('/content/drive')

PROJECT_ROOT = '/content/drive/MyDrive/Thesis'
SILVER_PATH  = os.path.join(PROJECT_ROOT, 'Silver')

TARGET_CRS   = 'EPSG:32632'
WGS84_CRS    = 'EPSG:4326'
H3_RESOLUTION = 9

print(f"H3 resolution:  {H3_RESOLUTION}")
print(f"Approx cell area at resolution 9: ~0.1 km²")
print(f"Approx centroid-to-edge distance: ~175 metres")
print(f"Analysis CRS:   {TARGET_CRS}")
print(f"Notebook started: {datetime.datetime.now().strftime('%Y-%m-%d %H:%M')}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
H3 resolution:  9
Approx cell area at resolution 9: ~0.1 km²
Approx centroid-to-edge distance: ~175 metres
Analysis CRS:   EPSG:32632
Notebook started: 2026-04-26 10:23


#Cell 4

In [ ]:
print("Loading Silver layer inputs...")
print()

study_area = gpd.read_file(
    os.path.join(SILVER_PATH, 'silver_study_area.geojson')
)
print(f"Study area:    {len(study_area)} feature, CRS: {study_area.crs}")

cafes = gpd.read_file(
    os.path.join(SILVER_PATH, 'silver_cafes.geojson')
)
print(f"Cafés:         {len(cafes)} features, CRS: {cafes.crs}")

viable_mask = gpd.read_file(
    os.path.join(SILVER_PATH, 'silver_viable_mask.geojson')
)
print(f"Viable mask:   {len(viable_mask)} feature, CRS: {viable_mask.crs}")

# Assert viable_mask contains exactly one dissolved feature. If NB2's dissolve
# failed partially and wrote a multi-row GDF, viable_mask.geometry.iloc[0] in
# Cell 7 silently uses only the first polygon, causing the viability filter to
# miss viable zones from subsequent rows.
assert len(viable_mask) == 1, (
    f"silver_viable_mask.geojson should contain exactly 1 dissolved feature, "
    f"found {len(viable_mask)}. Re-run NB2 Cell 10 to regenerate the viable mask."
)

# Extract the single unified viable zone polygon for spatial operations
# Note: viable_polygon in WGS84 was removed to prevent accidental usage
# in downstream area calculations. viable_union_geom in EPSG:32632
# is extracted in Cell 7 for accurate overlap computation.
print()
print("Silver inputs loaded successfully")

Loading Silver layer inputs...

Study area:    1 feature, CRS: EPSG:4326
Cafés:         1728 features, CRS: EPSG:4326
Viable mask:   1 feature, CRS: EPSG:4326

Silver inputs loaded successfully


#Cell 5

In [ ]:
# Generate all H3 cells at resolution 9 that cover the study area
#
# H3 library requires coordinates in WGS84 lat/lon — not EPSG:32632
# So we reproject the study area to WGS84 for the grid generation step,
# generate the cells, then immediately work back in EPSG:32632 for all
# spatial operations

print(f"Generating H3 grid at resolution {H3_RESOLUTION}...")
print()
print(f"  h3 version: {h3.__version__}")

study_area_wgs84    = study_area.to_crs(WGS84_CRS)
study_polygon_wgs84 = study_area_wgs84.dissolve().geometry.iloc[0]

# Warn explicitly if the dissolved study boundary
# is a MultiPolygon (e.g. due to disjoint neighbourhood extents or topology
# gaps). h3.geo_to_cells accepts MultiPolygon via __geo_interface__ in h3-py v4
# but the user should know how many parts are being covered, as internal voids
# produce fewer cells than a contiguous polygon of equal total area.
if study_polygon_wgs84.geom_type == 'MultiPolygon':
    n_parts = len(study_polygon_wgs84.geoms)
    print(f"  NOTE: study polygon is MultiPolygon with {n_parts} parts — "
          f"h3.geo_to_cells will cover all parts.")
    print(f"  Internal voids (if any) will not be covered; "
          f"verify total cell count is within expected range (~1,700–1,800 cells).")

assert study_polygon_wgs84.geom_type in ('Polygon', 'MultiPolygon'), \
    f"Unexpected geometry type: {study_polygon_wgs84.geom_type}"
study_geojson = mapping(study_polygon_wgs84)

h3_cells = h3.geo_to_cells(study_geojson, H3_RESOLUTION)

print(f"  H3 cells generated: {len(h3_cells)}")
print(f"  Sample cell ID:     {list(h3_cells)[0]}")

# Hard assert on cell count: a wrong count corrupts the entire feature matrix.
# Consistent with the assert pattern used for all other structural guards.
EXPECTED_CELLS_MIN = 1_700
EXPECTED_CELLS_MAX = 1_800
assert EXPECTED_CELLS_MIN <= len(h3_cells) <= EXPECTED_CELLS_MAX, (
    f"Cell count {len(h3_cells)} outside expected range "
    f"[{EXPECTED_CELLS_MIN}, {EXPECTED_CELLS_MAX}] — "
    f"check study area dissolve, H3_RESOLUTION, or boundary geometry."
)
print(f"  Cell count assertion passed: {len(h3_cells)} cells within "
      f"[{EXPECTED_CELLS_MIN}, {EXPECTED_CELLS_MAX}].")

Generating H3 grid at resolution 9...

  h3 version: 4.4.2
  H3 cells generated: 1743
  Sample cell ID:     891f99cd417ffff
  Cell count assertion passed: 1743 cells within [1700, 1800].


#Cell 6

In [ ]:
print("Building H3 grid GeoDataFrame...")

rows = []
for cell_id in h3_cells:
    boundary = h3.cell_to_boundary(cell_id)

    polygon_coords = [(lon, lat) for lat, lon in boundary]
    polygon        = Polygon(polygon_coords)

    rows.append({
        'h3_id':    cell_id,
        'geometry': polygon
    })

h3_grid = gpd.GeoDataFrame(rows, geometry='geometry', crs=WGS84_CRS)

h3_grid = h3_grid.to_crs(TARGET_CRS)

h3_grid['centroid_x'] = h3_grid.geometry.centroid.x
h3_grid['centroid_y'] = h3_grid.geometry.centroid.y

print(f"  Grid shape:         {h3_grid.shape}")
print(f"  CRS:                {h3_grid.crs}")
print(f"  Sample centroid X:  {h3_grid['centroid_x'].iloc[0]:.2f} m")
print(f"  Sample centroid Y:  {h3_grid['centroid_y'].iloc[0]:.2f} m")
print()
print(h3_grid.head(3))

Building H3 grid GeoDataFrame...
  Grid shape:         (1743, 4)
  CRS:                EPSG:32632
  Sample centroid X:  506645.88 m
  Sample centroid Y:  5036553.67 m

             h3_id                                           geometry  \
0  891f99cd417ffff  POLYGON ((506462.59 5036627.349, 506478.912 50...   
1  891f995666bffff  POLYGON ((517964.673 5030481.686, 517981.117 5...   
2  891f99cf303ffff  POLYGON ((507366.379 5038545.403, 507382.709 5...   

      centroid_x    centroid_y  
0  506645.876106  5.036554e+06  
1  518148.200244  5.030408e+06  
2  507549.664071  5.038472e+06  


#Cell 7

In [ ]:
# Apply the binary viability filter using proportional area overlap
#
# Previous approach (centroid-based):
#   A cell was viable only if its centroid fell within a viable zone.
#   Problem: a cell whose centroid is 10m inside a park but has 80% of
#   its area covered by viable urban fabric was discarded entirely.
#
# Updated approach (proportional overlap):
#   A cell is viable if at least VIABILITY_THRESHOLD of its total area
#   is covered by viable urban fabric polygons from Urban Atlas.
#   This recovers cells on the edges of parks, railways, and industrial
#   zones that have genuine commercial potential on their viable side.
#
# Viable geometry source:
#   silver_viable_mask.geojson is the single authoritative source for the
#   viable union geometry — it is the Silver layer contract output produced
#   by NB2 Cell 10. We load it directly rather than re-reading and
#   re-dissolving silver_urban_atlas.geojson, which would risk divergence
#   if code-type matching logic ever differs between notebooks.
#
# Threshold choice:
#   20% minimum overlap is consistent with urban planning practice —
#   a cell needs at least one fifth of its area to be viable urban
#   fabric before it is considered a candidate location.
#   This is documented as a methodological choice in the thesis.
#
# Clip h3_grid to the study boundary BEFORE computing cell_area_m2 and
# viable_overlap_ratio.
#
# Without clipping, cells that straddle the study boundary carry their full
# hexagon area (~105,000 m²) as the denominator. This understates
# viable_overlap_ratio for boundary cells whose within-study portion is
# predominantly viable — some cells were erroneously filtered as non-viable
# when their within-study viable fraction would exceed the 20% threshold.
# After clipping, cell_area_m2 reflects only the within-study portion, so
# the viability ratio and area assertions are both correct.

print("Applying proportional overlap viability filter...")
print(f"  Total H3 cells before filter: {len(h3_grid)}")

VIABILITY_THRESHOLD = 0.20  # 20%

# Reproject study area to TARGET_CRS for clipping
study_area_32632 = study_area.to_crs(TARGET_CRS)

# Assert study_area_32632 has a valid CRS and is in TARGET_CRS before passing
# to gpd.clip(). If silver_study_area.geojson
# has a missing/incorrect CRS embedded (can happen with some GeoJSON writers),
# .to_crs(TARGET_CRS) silently uses the wrong source CRS and produces a
# spatially incorrect clip without raising an error.
assert study_area_32632.crs is not None and study_area_32632.crs.to_epsg() == 32632, (
    f"study_area_32632 CRS unexpected: {study_area_32632.crs} — "
    f"expected EPSG:32632. Check silver_study_area.geojson CRS metadata."
)

# Clip h3_grid to the study boundary — boundary-straddling cells have their
# geometry (and thus cell_area_m2) reduced to the within-study portion only.
h3_grid_clipped = gpd.clip(h3_grid, study_area_32632)

# Preserve original h3_id index alignment (clip may reorder rows)
h3_grid_clipped = h3_grid_clipped.reset_index(drop=True)

print(f"  Cells after clip to study boundary: {len(h3_grid_clipped)}")

# Compute area from clipped geometries — boundary cells will be smaller than
# interior cells. This is expected and correct.
h3_grid_clipped['cell_area_m2'] = h3_grid_clipped.geometry.area

# Drop sliver cells introduced by gpd.clip() before computing
# viable_overlap_ratio. A sliver cell (e.g. 10,000–20,000 m²)
# has near-total viable coverage by area fraction, so its viable_overlap_ratio
# approaches 1.0 and it passes the viability threshold. Its cell_area_m2 then
# enters NB4 as a denominator for per-area features, producing inflated values.
# This mirrors the sliver-drop pattern in NB2 Cell 9 for census tracts.
SLIVER_THRESHOLD_M2 = 5_000
n_before_sliver = len(h3_grid_clipped)
h3_grid_clipped = h3_grid_clipped[
    h3_grid_clipped['cell_area_m2'] >= SLIVER_THRESHOLD_M2
].copy().reset_index(drop=True)
n_slivers_dropped = n_before_sliver - len(h3_grid_clipped)
print(f"  Sliver cells dropped (area < {SLIVER_THRESHOLD_M2:,} m²): {n_slivers_dropped}")
if n_slivers_dropped > 0:
    print(f"  NOTE: {n_slivers_dropped} sliver cell(s) from gpd.clip() removed "
          f"before viability computation to prevent distorted per-area features in NB4.")

# Area assertion — lower bound updated to match sliver threshold.
_area_min = h3_grid_clipped['cell_area_m2'].min()
_area_max = h3_grid_clipped['cell_area_m2'].max()
assert h3_grid_clipped['cell_area_m2'].between(SLIVER_THRESHOLD_M2, 120_000).all(), (
    "Unexpected cell area — possible degenerate H3 geometry. "
    f"Min: {_area_min:.0f} m², Max: {_area_max:.0f} m² "
    f"(expected range: {SLIVER_THRESHOLD_M2:,}–120,000 m² for H3 res-9 "
    f"including clipped boundary cells)"
)

n_small_cells = (h3_grid_clipped['cell_area_m2'] < 80_000).sum()
if n_small_cells > 0:
    print(f"  NOTE: {n_small_cells} boundary-clipped cell(s) have area < 80,000 m² "
          f"(min={_area_min:.0f} m²) — these are cells clipped to the study boundary, "
          f"as expected. Not an error.")
else:
    print(f"  cell_area_m2 check passed: all cells within 80,000–120,000 m².")
print(f"  cell_area_m2 stats: mean={h3_grid_clipped['cell_area_m2'].mean():.0f} m², "
      f"min={_area_min:.0f} m², max={_area_max:.0f} m²")

# Load the authoritative viable mask from the Silver layer
# viable_mask is already loaded in Cell 4; confirm it is in TARGET_CRS
if viable_mask.crs.to_epsg() != 32632:
    viable_mask = viable_mask.to_crs(TARGET_CRS)

# Use the single dissolved viable polygon — identical to what NB2 stored
viable_union_geom = viable_mask.geometry.iloc[0]
# Confirmed: viable_union_geom is safely in EPSG:32632 at this stage
# Area calculations below will correctly be in square meters.

# Repair any self-intersections that may have been introduced by the
# dissolve() in NB2 — buffer(0) is the canonical Shapely fix.
# A self-intersecting polygon causes intersection() to return empty or
# invalid geometries silently, marking cells non-viable without error.
viable_union_geom = viable_union_geom.buffer(0)
assert viable_union_geom.is_valid, (
    "viable_union_geom is invalid after buffer(0) repair — "
    "check NB2 dissolve output in silver_viable_mask.geojson"
)

print(f"  Viable mask loaded from silver_viable_mask.geojson")
print(f"  Viable union geometry valid: {viable_union_geom.is_valid}")
print(f"  Viable union area: {viable_union_geom.area / 1_000_000:.2f} km²")

def compute_viable_overlap_ratio(cell_geom, viable_geom=viable_union_geom):
    """
    Overlap ratio = intersection(cell, viable) / cell.area.
    cell.area is the clipped (within-study) area, so boundary cells
    are assessed against their within-study portion only.
    """
    try:
        intersection = cell_geom.intersection(viable_geom)
        if intersection.is_empty:
            return 0.0
        return intersection.area / cell_geom.area
    except Exception as e:
        warnings.warn(
            f"Intersection failed for cell centroid "
            f"({cell_geom.centroid.x:.1f}, {cell_geom.centroid.y:.1f}): {e}"
        )
        return 0.0

h3_grid_clipped['viable_overlap_ratio'] = h3_grid_clipped['geometry'].apply(
    compute_viable_overlap_ratio
)

# Post-loop exception audit: any cell that returned 0.0 AND whose centroid
# is geometrically inside the viable union is a candidate silent failure.
n_zero_overlap = (h3_grid_clipped['viable_overlap_ratio'] == 0.0).sum()
n_centroid_inside = h3_grid_clipped[h3_grid_clipped['viable_overlap_ratio'] == 0.0].geometry.apply(
    lambda g: viable_union_geom.contains(g.centroid)
).sum()

if n_centroid_inside > 0:
    print(
        f"  WARNING: {n_centroid_inside} cell(s) have viable_overlap_ratio=0.0 "
        f"but their centroid falls inside the viable union — possible silent "
        f"intersection failures. Review warnings above for details."
    )
else:
    print(f"  Intersection exception audit: 0 suspicious zero-overlap cells detected.")

# Tight interior-cell assertion
interior_mask = h3_grid_clipped['viable_overlap_ratio'] == 1.0
if interior_mask.any():
    assert h3_grid_clipped.loc[interior_mask, 'cell_area_m2'].between(80_000, 120_000).all(), (
        "Interior cell area outside expected range [80,000–120,000 m²] — "
        "check that buffer(0) repair on viable_union_geom did not clip cell polygons. "
        f"Min interior area: {h3_grid_clipped.loc[interior_mask, 'cell_area_m2'].min():.0f} m², "
        f"Max interior area: {h3_grid_clipped.loc[interior_mask, 'cell_area_m2'].max():.0f} m²"
    )
    print(f"  Interior cell area assertion passed "
          f"({interior_mask.sum()} fully-interior cells within 80,000–120,000 m²).")

h3_grid_clipped['is_viable'] = (
    h3_grid_clipped['viable_overlap_ratio'] >= VIABILITY_THRESHOLD
)

# Propagate clipped columns back onto h3_grid so downstream cells (Cell 9
# visualisation and log) continue to reference h3_grid correctly.
# Cells removed by gpd.clip() (outside the study boundary) are dropped —
# this is correct; they should not appear in any output.
h3_grid = h3_grid_clipped.copy()

# Recompute centroid_x / centroid_y on h3_grid after reassignment from
# h3_grid_clipped. Boundary cells now have
# their geometry truncated to the within-study portion, so the centroid
# values computed from full hexagons in Cell 6 are stale for those cells.
# Recomputing here ensures centroid_x / centroid_y reflect the clipped
# geometry anchor, consistent with cell_area_m2 and viable_overlap_ratio.
h3_grid['centroid_x'] = h3_grid.geometry.centroid.x
h3_grid['centroid_y'] = h3_grid.geometry.centroid.y

n_viable     = h3_grid['is_viable'].sum()
n_non_viable = (~h3_grid['is_viable']).sum()

print(f"\n  Viability threshold:      {VIABILITY_THRESHOLD*100:.0f}% overlap")
print(f"  Viable cells:             {n_viable}")
print(f"  Non-viable cells:         {n_non_viable}")
print(f"  Viability rate:           {n_viable / len(h3_grid) * 100:.1f}%")

print(f"\n  Overlap ratio distribution:")
print(f"    0%:        "
      f"{(h3_grid['viable_overlap_ratio'] == 0).sum()} cells")
print(f"    1-{int(VIABILITY_THRESHOLD*100)-1}%:     "
      f"{((h3_grid['viable_overlap_ratio'] > 0) & (h3_grid['viable_overlap_ratio'] < VIABILITY_THRESHOLD)).sum()} cells")
print(f"    {VIABILITY_THRESHOLD*100:.0f}-49%:    "
      f"{((h3_grid['viable_overlap_ratio'] >= VIABILITY_THRESHOLD) & (h3_grid['viable_overlap_ratio'] < 0.50)).sum()} cells")
print(f"    50-79%:    "
      f"{((h3_grid['viable_overlap_ratio'] >= 0.50) & (h3_grid['viable_overlap_ratio'] < 0.80)).sum()} cells")
print(f"    80-100%:   "
      f"{(h3_grid['viable_overlap_ratio'] >= 0.80).sum()} cells")

h3_viable = h3_grid[h3_grid['is_viable']].copy().reset_index(drop=True)

# Propagate recomputed centroids into h3_viable.
# h3_viable is a subset of h3_grid after the copy above, so its
# centroid_x / centroid_y already reflect clipped geometry (inherited from the
# h3_grid recomputation two blocks above). The explicit reassignment below
# is a defensive guard in case of any future reordering of the subset step.
h3_viable['centroid_x'] = h3_viable.geometry.centroid.x
h3_viable['centroid_y'] = h3_viable.geometry.centroid.y

print(f"\n  Analysis backbone: {len(h3_viable)} viable candidate cells")

Applying proportional overlap viability filter...
  Total H3 cells before filter: 1743
  Cells after clip to study boundary: 1743
  Sliver cells dropped (area < 5,000 m²): 0
  NOTE: 68 boundary-clipped cell(s) have area < 80,000 m² (min=34185 m²) — these are cells clipped to the study boundary, as expected. Not an error.
  cell_area_m2 stats: mean=102429 m², min=34185 m², max=104791 m²
  Viable mask loaded from silver_viable_mask.geojson
  Viable union geometry valid: True
  Viable union area: 63.17 km²
  Intersection exception audit: 0 suspicious zero-overlap cells detected.

  Viability threshold:      20% overlap
  Viable cells:             971
  Non-viable cells:         772
  Viability rate:           55.7%

  Overlap ratio distribution:
    0%:        386 cells
    1-19%:     386 cells
    20-49%:    324 cells
    50-79%:    478 cells
    80-100%:   169 cells

  Analysis backbone: 971 viable candidate cells


#Cell 8

In [ ]:
print("Assigning cafés to H3 cells...")

for col in ['cafe_count', 'label']:
    if col in h3_viable.columns:
        h3_viable = h3_viable.drop(columns=[col])

# cafes is loaded from silver_cafes.geojson in Cell 4,
# which is in WGS84 (GeoJSON RFC 7946). The .to_crs(WGS84_CRS) call
# is therefore always a no-op. The comment previously said "Reproject to WGS84
# for H3 assignment" implying cafes was in EPSG:32632, which was misleading
# and caused reviewers to question the CRS used in the viability check in
# Cell 7 (which uses viable_union_geom in EPSG:32632 separately).
# The assertion makes the actual CRS state explicit and self-documenting.
assert cafes.crs.to_epsg() == 4326, (
    f"cafes CRS should be WGS84 at this stage, got {cafes.crs}"
)
# cafes loaded from silver_cafes.geojson are already in WGS84 —
# .to_crs() is a no-op but kept for explicitness.
cafes_wgs84 = cafes.to_crs(WGS84_CRS)

# Guard against non-Point geometries before the h3.latlng_to_cell() apply.
# Non-Point geometries produce AttributeError on
# .y / .x in some Shapely versions. NB2 converts all café geometries to
# centroids, but this guard catches any edge-case slip-through.
n_non_point = (cafes_wgs84.geometry.geom_type != 'Point').sum()
if n_non_point > 0:
    print(f"  WARNING: {n_non_point} non-Point café geometry(ies) found — dropping "
          f"before H3 assignment. These should have been centroided in NB2.")
cafes_wgs84 = cafes_wgs84[cafes_wgs84.geometry.geom_type == 'Point'].copy()

def assign_h3_cell(geometry, resolution=H3_RESOLUTION):
    try:
        return h3.latlng_to_cell(
            geometry.y,   # latitude
            geometry.x,   # longitude
            resolution
        )
    except Exception as e:
        warnings.warn(
            f"H3 assignment failed for ({geometry.x:.1f}, {geometry.y:.1f}): {e}"
        )
        return None

cafes_wgs84['h3_id'] = cafes_wgs84.geometry.apply(assign_h3_cell)

# Capture pre-drop length to report silent failures.
cafes_wgs84_before_drop = len(cafes_wgs84)

cafes_wgs84 = cafes_wgs84.dropna(subset=['h3_id'])

# Warn if any cafés were silently dropped — corrupts positive-class label counts
n_failed = cafes_wgs84_before_drop - len(cafes_wgs84)
if n_failed > 0:
    print(f"  WARNING: {n_failed} café(s) failed H3 assignment and were dropped.")
    print(f"  This may indicate coordinates outside valid H3 range or library errors.")
    print(f"  Verify input CRS is WGS84 and coordinates are within Milan bounds.")

cafe_counts = (
    cafes_wgs84.groupby('h3_id')
    .size()
    .reset_index(name='cafe_count')
)

# Reconciliation: separately count cafés outside the grid entirely and
# cafés inside the grid but in non-viable cells.
# Previously both groups were collapsed into one "non-viable" count, which
# misattributed out-of-boundary cafés to the non-viable category and made
# the audit trail misleading.
n_outside_grid = len(
    cafes_wgs84[~cafes_wgs84['h3_id'].isin(h3_grid['h3_id'])]
)
n_in_nonviable = len(
    cafes_wgs84[
        cafes_wgs84['h3_id'].isin(
            h3_grid.loc[~h3_grid['is_viable'], 'h3_id']
        )
    ]
)
print(f"  Cafés outside grid boundary (no cell assigned):  {n_outside_grid}")
print(f"  Cafés in non-viable cells (excluded from label): {n_in_nonviable}")
print(f"  Cafés in viable cells (used for label):          "
      f"{len(cafes_wgs84) - n_outside_grid - n_in_nonviable}")

h3_viable = h3_viable.merge(cafe_counts, on='h3_id', how='left')
h3_viable['cafe_count'] = h3_viable['cafe_count'].fillna(0).astype(int)

POSITIVE_CLASS_THRESHOLD = 2
h3_viable['label'] = (
    h3_viable['cafe_count'] >= POSITIVE_CLASS_THRESHOLD
).astype(int)

n_positive   = h3_viable['label'].sum()
n_negative   = (h3_viable['label'] == 0).sum()
pct_positive = n_positive / len(h3_viable) * 100

print(f"  Cafés successfully assigned (post-drop):  {len(cafes_wgs84)}")
print(f"  Positive class threshold:     {POSITIVE_CLASS_THRESHOLD} cafés per cell")
print(f"  Cells with 2+ cafés:          {n_positive} (label=1)")
print(f"  Cells with fewer than 2:      {n_negative} (label=0)")
print(f"  Class imbalance ratio:        {pct_positive:.1f}% positive")
print()
print(f"  NOTE: Class imbalance will be addressed in Notebook 6")
print(f"  via stratified resampling before Random Forest training")

# Save café-to-cell index (with osmid for traceability)
cafe_index_path = os.path.join(SILVER_PATH, 'silver_cafe_cell_index.csv')

osmid_cols = [c for c in cafes_wgs84.columns if 'osmid' in c.lower()]
if osmid_cols:
    cafe_index_df = cafes_wgs84[['h3_id', osmid_cols[0]]].copy()
    cafe_index_df = cafe_index_df.rename(columns={osmid_cols[0]: 'osmid'})
elif cafes_wgs84.index.name == 'osmid':
    cafe_index_df = cafes_wgs84[['h3_id']].copy()
    cafe_index_df['osmid'] = cafes_wgs84.index
else:
    # osmid not present — index is used as fallback identifier
    cafe_index_df = cafes_wgs84[['h3_id']].copy()
    cafe_index_df['osmid'] = cafes_wgs84.index
    print("  WARNING: osmid column not found; using DataFrame index as osmid.")

cafe_index_df.to_csv(cafe_index_path, index=False)
print(f"  Café cell index saved: silver_cafe_cell_index.csv "
      f"({len(cafe_index_df)} rows, columns: {list(cafe_index_df.columns)})")

Assigning cafés to H3 cells...
  Cafés outside grid boundary (no cell assigned):  2
  Cafés in non-viable cells (excluded from label): 123
  Cafés in viable cells (used for label):          1603
  Cafés successfully assigned (post-drop):  1728
  Positive class threshold:     2 cafés per cell
  Cells with 2+ cafés:          345 (label=1)
  Cells with fewer than 2:      626 (label=0)
  Class imbalance ratio:        35.5% positive

  NOTE: Class imbalance will be addressed in Notebook 6
  via stratified resampling before Random Forest training
  Café cell index saved: silver_cafe_cell_index.csv (1728 rows, columns: ['h3_id', 'osmid'])


#Cell 9

In [ ]:
print("Saving H3 grid outputs...")
print()

# Export in WGS84 for GeoJSON compatibility
# NOTE: round_coordinates (which applied simplify(1e-6)) has been removed.
# H3 hexagon boundaries must NOT be simplified — even at 1e-6 degrees this
# modifies canonical H3 boundaries and introduces centroid displacement.
# GeoPandas' default GeoJSON writer rounds to 6 significant figures without
# modifying stored geometries, which is sufficient for file size management.
h3_grid_wgs84   = h3_grid.to_crs(WGS84_CRS)
h3_viable_wgs84 = h3_viable.to_crs(WGS84_CRS)

full_grid_path = os.path.join(SILVER_PATH, 'silver_h3_grid.geojson')
h3_grid_wgs84.to_file(full_grid_path, driver='GeoJSON')
print(f"  [Full grid]   silver_h3_grid.geojson "
      f"({len(h3_grid_wgs84)} cells)")

viable_grid_path = os.path.join(SILVER_PATH, 'silver_h3_viable.geojson')
h3_viable_wgs84.to_file(viable_grid_path, driver='GeoJSON')
print(f"  [Viable grid] silver_h3_viable.geojson "
      f"({len(h3_viable_wgs84)} cells)")

print()
print(f"Files saved to: {SILVER_PATH}")

Saving H3 grid outputs...

  [Full grid]   silver_h3_grid.geojson (1743 cells)
  [Viable grid] silver_h3_viable.geojson (971 cells)

Files saved to: /content/drive/MyDrive/Thesis/Silver


#Cell 10

In [ ]:
log_lines = [
    f"Thesis — Phase 2 · H3 Grid Generation Log",
    f"==========================================",
    f"Run date:       {datetime.datetime.now().strftime('%Y-%m-%d %H:%M')}",
    f"H3 resolution:  {H3_RESOLUTION}",
    f"Cell area:      ~0.1 km²",
    f"Analysis CRS:   {TARGET_CRS}",
    f"",
    f"Grid statistics:",
    f"  Total cells generated:    {len(h3_grid)}",
    f"  Viable cells:             {n_viable}",
    f"  Non-viable cells:         {n_non_viable}",
    f"  Viability rate:           {n_viable / len(h3_grid) * 100:.1f}%",
    f"",
    f"Class label statistics:",
    f"  Positive class threshold: {POSITIVE_CLASS_THRESHOLD} cafés per cell",
    f"  Positive cells (label=1): {n_positive}",
    f"  Negative cells (label=0): {n_negative}",
    f"  Positive class rate:      {pct_positive:.1f}%",
    f"",
    f"Outputs:",
    f"  silver_h3_grid.geojson",
    f"  silver_h3_viable.geojson",
    f"  silver_cafe_cell_index.csv",
]

log_path = os.path.join(SILVER_PATH, 'silver_h3_log.txt')
with open(log_path, 'w') as f:
    f.write('\n'.join(log_lines))

print('\n'.join(log_lines))
print(f"\nLog saved to: {log_path}")

Thesis — Phase 2 · H3 Grid Generation Log
Run date:       2026-04-26 10:28
H3 resolution:  9
Cell area:      ~0.1 km²
Analysis CRS:   EPSG:32632

Grid statistics:
  Total cells generated:    1743
  Viable cells:             971
  Non-viable cells:         772
  Viability rate:           55.7%

Class label statistics:
  Positive class threshold: 2 cafés per cell
  Positive cells (label=1): 345
  Negative cells (label=0): 626
  Positive class rate:      35.5%

Outputs:
  silver_h3_grid.geojson
  silver_h3_viable.geojson
  silver_cafe_cell_index.csv

Log saved to: /content/drive/MyDrive/Thesis/Silver/silver_h3_log.txt


#Cell 11

In [ ]:
# Visualize the H3 grid overlaid on the study area
# Green cells = viable (candidate café locations)
# Grey cells  = non-viable (excluded by binary filter)
# Red dots    = existing café locations

m = folium.Map(
    location=[45.4642, 9.1900],
    zoom_start=12,
    tiles='CartoDB positron'
)

# Non-viable cells — grey
non_viable_display = h3_grid_wgs84[~h3_grid_wgs84['is_viable']].head(500)
folium.GeoJson(
    non_viable_display.__geo_interface__,
    name='Non-viable cells',
    style_function=lambda x: {
        'fillColor': '#888888',
        'color':     '#666666',
        'weight':    0.3,
        'fillOpacity': 0.3
    }
).add_to(m)

# Viable cells — green
viable_display = h3_viable_wgs84.head(800)
folium.GeoJson(
    viable_display.__geo_interface__,
    name='Viable cells',
    style_function=lambda x: {
        'fillColor': '#1D9E75',
        'color':     '#0F6E56',
        'weight':    0.5,
        'fillOpacity': 0.4
    }
).add_to(m)

# Café locations
cafes_display = cafes.to_crs(WGS84_CRS).head(500)
for _, row in cafes_display.iterrows():
    if row.geometry is not None:
        folium.CircleMarker(
            location=[row.geometry.y, row.geometry.x],
            radius=3,
            color='#E8593C',
            fill=True,
            fill_opacity=0.8
        ).add_to(m)

folium.LayerControl().add_to(m)

print("Green  = viable H3 cells (candidate café locations)")
print("Grey   = non-viable H3 cells (excluded by binary filter)")
print("Red    = existing café locations from OSM")
m

Green  = viable H3 cells (candidate café locations)
Grey   = non-viable H3 cells (excluded by binary filter)
Red    = existing café locations from OSM


#Cell 12

In [ ]:
print("=" * 60)
print("NOTEBOOK 3 COMPLETE — H3 Grid Summary")
print("=" * 60)

expected_files = [
    'silver_h3_grid.geojson',
    'silver_h3_viable.geojson',
    'silver_cafe_cell_index.csv',
    'silver_h3_log.txt'
]

all_present = True
for filename in expected_files:
    filepath = os.path.join(SILVER_PATH, filename)
    status   = "OK" if os.path.exists(filepath) else "MISSING"
    if status == "MISSING":
        all_present = False
    print(f"  [{status}] {filename}")

print()
print(f"  Viable candidate cells:   {len(h3_viable)}")
print(f"  Positive class (café):    {n_positive}")
print(f"  Negative class (no café): {n_negative}")
print(f"  Class imbalance:          {pct_positive:.1f}% positive")

print()
if all_present:
    print("All outputs present. Ready to proceed to Notebook 4.")
    print("Next: feature engineering — computing spatial indicators")
    print("per H3 cell centroid.")
else:
    print("Some files missing. Review cells above before proceeding.")

NOTEBOOK 3 COMPLETE — H3 Grid Summary
  [OK] silver_h3_grid.geojson
  [OK] silver_h3_viable.geojson
  [OK] silver_cafe_cell_index.csv
  [OK] silver_h3_log.txt

  Viable candidate cells:   971
  Positive class (café):    345
  Negative class (no café): 626
  Class imbalance:          35.5% positive

All outputs present. Ready to proceed to Notebook 4.
Next: feature engineering — computing spatial indicators
per H3 cell centroid.
